# Orbit Wars icin Set Transformer not defteri

Bu notebook iki amaca hizmet eder:

1. Set Transformer mantigini teorik olarak anlamak.
2. Bu repodaki `obs_builder.py` ve `model.py` uzerinden Orbit Wars policy/value modelini calistirmak.

Not: Model kodu PyTorch ister. Kernel ortaminda `torch` yoksa ilk import hucreleri acik bir hata mesaji verir.

## 1. Neden Set Transformer?

Orbit Wars gozlemi dogal olarak bir grid degil, bir **set** problemidir:

- Planetler: sirasi anlamsal olmayan degisken sayida nesne.
- Fleetler: sirasi anlamsal olmayan degisken sayida nesne.
- Global durum: turn, toplam gemi, toplam production gibi ozet bilgiler.

Klasik MLP'ye planetleri `[planet_0, planet_1, ...]` diye verirsek model slot sirasina fazla baglanabilir. Set Transformer yaklasimi ise her nesneyi token gibi isler ve self-attention ile tokenlar arasi iliskiyi ogrenmesini saglar.

Temel kavramlar:

- **Permutation equivariance**: Girdi setindeki elemanlarin sirasi degisirse, cikti tokenlari da ayni sekilde yer degistirir. Planet basina source/ship logitleri icin bu ozellik uygundur.
- **Permutation invariance**: Girdi setindeki elemanlarin sirasi degisse bile tekil global cikti degismez. Value head icin bu ozellik uygundur.
- **Self-attention**: Her token diger tokenlara bakarak kendi temsilini gunceller. Bir planet, yakindaki enemy planetleri, gelen fleetleri ve global durumu ayni encoder icinde gorebilir.
- **Masking**: Padding tokenlari ve illegal action slotlari softmax disina itilir.

## 2. Klasik Set Transformer bloklari ve bu repodaki karsiligi

Set Transformer literaturunde sik kullanilan bloklar:

- **SAB (Self-Attention Block)**: Set elemanlari birbirine attention uygular.
- **ISAB (Induced SAB)**: Cok buyuk setlerde maliyeti dusurmek icin az sayida inducing point kullanir.
- **PMA (Pooling by Multihead Attention)**: Setten sabit sayida global cikti tokeni uretir.

Bu repo birebir paper implementasyonu yerine Orbit Wars'a uyarlanmis pratik bir varyant kullaniyor:

- `obs_builder.py` planet ve fleetleri sabit `P_max`, `F_max` boyutuna pad ediyor.
- Token dizilimi: `[planet_0..planet_Pmax, fleet_0..fleet_Fmax, GLOBAL]`.
- `model.py` icindeki `TransformerBlock`, SAB benzeri self-attention blogu.
- `GLOBAL` token, PMA benzeri global ozet rolu oynuyor ve value head'e gidiyor.
- `edge_bias`, attention skorlarina eklenen geometrik on bilgi: mesafe, dx, dy.
- Action head'leri planet tokenlarindan source, target ve ship-bin secimleri uretiyor.

In [4]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "model.py").exists() else NOTEBOOK_DIR / "Orbit_Wars"
if not (PROJECT_ROOT / "model.py").exists():
    raise FileNotFoundError("model.py bulunamadi. Notebook'u repo kokunden veya Orbit_Wars klasorunden calistirin.")

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/Users/doruksarigun/Kaggle/orbit-wars/Orbit_Wars')

In [5]:
try:
    import torch
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Bu notebook OrbitWarsPolicy modelini calistirmak icin PyTorch ister. "
        "Kernel ortaminda torch kurulu degil. Uygun Python ortamina PyTorch kurup tekrar calistirin."
    ) from exc

from obs_builder import build_observation, decode_action
from model import OrbitWarsPolicy, pack_observations
from feature_extraction import parse_planets

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## 3. Toy Orbit Wars gozlemi

Asagidaki ornek Kaggle env'e bagli degil. Repo testlerindeki formata yakin kucuk bir observation kullaniyoruz. Boylece feature builder ve model akisini izole sekilde deneyebiliriz.

In [6]:
toy_obs = {
    "planets": [
        [0, 0, 70.0, 70.0, 3.0, 100, 3],  # my planet
        [1, 0, 80.0, 70.0, 2.0, 50, 2],   # my planet
        [2, 1, 30.0, 30.0, 2.0, 80, 2],   # enemy planet
        [3, -1, 20.0, 80.0, 2.0, 10, 1],  # neutral planet
    ],
    "fleets": [
        [10, 1, 60.0, 60.0, 0.0, 2, 25],
    ],
    "initial_planets": [
        [0, 0, 70.0, 70.0, 3.0, 100, 3],
        [1, 0, 80.0, 70.0, 2.0, 50, 2],
        [2, 1, 30.0, 30.0, 2.0, 80, 2],
        [3, -1, 20.0, 80.0, 2.0, 10, 1],
    ],
    "comet_planet_ids": [],
    "comets": [],
}

obs = build_observation(
    toy_obs,
    player=0,
    angular_velocity=0.05,
    turn=12,
    p_max=8,
    f_max=4,
)

print(obs.meta)
print("planet_feats", obs.planet_feats.shape)
print("fleet_feats", obs.fleet_feats.shape)
print("global_feats", obs.global_feats.shape)
print("edge_bias", obs.edge_bias.shape)
print("source_mask", obs.source_mask.astype(int))
print("planet_ids", obs.planet_ids)

{'n_planets': 4, 'n_fleets': 1, 'incoming_self': 0, 'incoming_other': 0, 'p_max': 8, 'f_max': 4}
planet_feats (8, 15)
fleet_feats (4, 11)
global_feats (12,)
edge_bias (13, 13, 3)
source_mask [1 1 0 0 0 0 0 0]
planet_ids [ 0  1  2  3 -1 -1 -1 -1]


## 4. Observation tensorlerinin anlami

`build_observation` su kontrati uretir:

- `planet_feats`: `(P_max, 15)` planet ozellikleri.
- `fleet_feats`: `(F_max, 11)` fleet ozellikleri.
- `global_feats`: `(12,)` global ozet.
- `planet_mask`, `fleet_mask`: gercek token mi padding mi?
- `source_mask`: bu planetten launch yapilabilir mi?
- `edge_bias`: `(T, T, 3)`, burada `T = P_max + F_max + 1`; tokenlar arasi mesafe ve yon bilgisi.

Model acisindan kritik nokta: token sirasi sadece slot eslestirmesi icin kullaniliyor; attention ile her token digerlerini set gibi okuyabiliyor, padding ise maskeleniyor.

In [7]:
batch = pack_observations([obs, obs], device=device)
{k: tuple(v.shape) for k, v in batch.items()}

{'planet_feats': (2, 8, 15),
 'fleet_feats': (2, 4, 11),
 'global_feats': (2, 12),
 'edge_bias': (2, 13, 13, 3),
 'planet_mask': (2, 8),
 'fleet_mask': (2, 4),
 'source_mask': (2, 8)}

In [8]:
net = OrbitWarsPolicy(
    p_max=8,
    f_max=4,
    d_model=64,
    n_heads=4,
    n_layers=2,
    n_ship_bins=5,
).to(device)

out = net(**batch)
print("src_logits", tuple(out.src_logits.shape))       # B, P_max + 1; son slot NO-OP
print("tgt_logits", tuple(out.tgt_logits.shape))       # B, P_max, P_max
print("ship_logits", tuple(out.ship_logits.shape))     # B, P_max, n_ship_bins
print("value", tuple(out.value.shape))                 # B
print("tokens", tuple(out.tokens.shape))               # B, T, d_model

src_logits (2, 9)
tgt_logits (2, 8, 8)
ship_logits (2, 8, 5)
value (2,)
tokens (2, 13, 64)


## 5. Maskelerin dogru calistigini kontrol edelim

Toy observation'da sadece planet slotlari `0` ve `1` bize ait oldugu icin source policy bu iki slot ve NO-OP disina probability vermemeli. Target policy de padding planetleri ve self-target secimini maskeliyor.

In [9]:
src_probs = torch.softmax(out.src_logits, dim=-1).detach().cpu()[0]
print("source probabilities, slots 0..P and NO-OP at last:")
for i, p in enumerate(src_probs.tolist()):
    name = "NO-OP" if i == net.p_max else f"planet_slot_{i}"
    print(f"{name:>13}: {p:.4f}")

tgt_probs_src0 = torch.softmax(out.tgt_logits[0, 0], dim=-1).detach().cpu()
print("\ntarget probabilities given source slot 0:")
for i, p in enumerate(tgt_probs_src0.tolist()):
    print(f"target_slot_{i}: {p:.4f}")

source probabilities, slots 0..P and NO-OP at last:
planet_slot_0: 0.4446
planet_slot_1: 0.4320
planet_slot_2: 0.0000
planet_slot_3: 0.0000
planet_slot_4: 0.0000
planet_slot_5: 0.0000
planet_slot_6: 0.0000
planet_slot_7: 0.0000
        NO-OP: 0.1234

target probabilities given source slot 0:
target_slot_0: 0.0000
target_slot_1: 0.3426
target_slot_2: 0.3259
target_slot_3: 0.3315
target_slot_4: 0.0000
target_slot_5: 0.0000
target_slot_6: 0.0000
target_slot_7: 0.0000


## 6. Action factorization

Model action'u uc parcaya ayiriyor:

1. `src`: Hangi planetten cikis yapacagiz? Ekstra `P_max` slotu NO-OP.
2. `tgt`: Secilen source icin hangi planet hedef?
3. `ship_bin`: Source gemilerinin hangi fraksiyonu gonderilecek? Varsayilan binler `0.1, 0.25, 0.5, 0.75, 1.0`.

Bu ayrim full action uzayini tek dev softmax yapmak yerine daha kucuk ve maskelenebilir kosullu dagilimlara boler.

In [10]:
net.eval()
with torch.no_grad():
    action = net.act(out, deterministic=True)

{k: v.detach().cpu().tolist() for k, v in action.items()}

{'src': [0, 0],
 'tgt': [1, 1],
 'ship_bin': [2, 2],
 'is_noop': [False, False],
 'log_prob': [-2.9266040325164795, -2.9266040325164795]}

In [11]:
# Untrained modelin deterministic secimi NO-OP olabilir. Decode mantigini net gormek
# icin burada elle gecerli bir source/target/bin secimi kullaniyoruz.
planets = parse_planets(toy_obs)
decoded = decode_action(
    obs,
    src_slot=0,
    tgt_slot=2,
    ship_bin=2,
    planets=planets,
    angular_velocity=0.05,
)
decoded

[0, -1.9425882279853135, 50]

## 7. Tek mini training step

Asagidaki hucre gercek RL egitimi degil; sadece gradient akisini ve head loss'larini gosteren minimum ornektir. Gercek PPO/A2C tarafinda advantage, return ve sampled log-prob kullanilir.

Burada sahte label'lar:

- source: slot `0`
- target: slot `2`
- ship bin: `2`, yani varsayilan fraksiyonlarda `0.5`
- value target: `1.0`

In [12]:
net.train()
optimizer = torch.optim.AdamW(net.parameters(), lr=3e-4)

out = net(**batch)
B = batch["planet_feats"].shape[0]

src_label = torch.zeros(B, dtype=torch.long, device=device)
tgt_label = torch.full((B,), 2, dtype=torch.long, device=device)
ship_label = torch.full((B,), 2, dtype=torch.long, device=device)
value_target = torch.ones(B, dtype=torch.float32, device=device)

src_loss = F.cross_entropy(out.src_logits, src_label)
selected_tgt_logits = out.tgt_logits[torch.arange(B, device=device), src_label]
tgt_loss = F.cross_entropy(selected_tgt_logits, tgt_label)
selected_ship_logits = out.ship_logits[torch.arange(B, device=device), src_label]
ship_loss = F.cross_entropy(selected_ship_logits, ship_label)
value_loss = F.mse_loss(out.value, value_target)

loss = src_loss + tgt_loss + ship_loss + 0.5 * value_loss
optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
optimizer.step()

print({
    "loss": round(loss.item(), 4),
    "src_loss": round(src_loss.item(), 4),
    "tgt_loss": round(tgt_loss.item(), 4),
    "ship_loss": round(ship_loss.item(), 4),
    "value_loss": round(value_loss.item(), 4),
    "grad_norm_before_clip": round(float(grad_norm), 4),
})

{'loss': 3.4413, 'src_loss': 0.8106, 'tgt_loss': 1.1212, 'ship_loss': 1.0448, 'value_loss': 0.9294, 'grad_norm_before_clip': 9.1685}


## 8. Kod okuma haritasi

Bu notebook'u calistirdiktan sonra su dosyalari birlikte okunmali:

- `obs_builder.py`: raw env observation'dan set token ozellikleri, maskeler ve edge bias uretir.
- `feature_extraction.py`: planet/fleet wrapper'lari, polar koordinatlar ve hareket tahminleri.
- `navigation.py`: secilen action'i gercek launch acisina ceviren geometri.
- `model.py`: attention bias'li Set Transformer encoder, action head'leri ve value head.
- `tests/test_model.py`: shape, mask ve gradient invariant'larini dogrulayan testler.

Pratik debug sirasi:

1. `obs.meta`, `planet_ids`, `source_mask` dogru mu?
2. `edge_bias` shape'i `P_max + F_max + 1` ile uyumlu mu?
3. `src_probs` invalid source slotlarina probability veriyor mu?
4. `tgt_logits[src]` padding veya self-target secimini disliyor mu?
5. `decode_action` `None` donduruyorsa sebep invalid slot mu, yoksa `compute_launch` gecerli rota bulamiyor mu?